In [2]:
!pip install -q sentence-transformers scikit-learn

In [3]:
%%writefile guided_suggestions.py

Writing guided_suggestions.py


In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from task2_chunks import docs, make_chunks


# Step 1: Create chunks

records = make_chunks(docs, "sentence")

chunk_texts = []

for record in records:
    chunk_texts.append(record["text"])

print("Total chunks:", len(records))


# Step 2: Prepare TF-IDF

vectorizer = TfidfVectorizer()

tfidf_data = vectorizer.fit_transform(chunk_texts)


# Step 3: Load embedding model

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

document_embeddings = model.encode(
    chunk_texts,
    normalize_embeddings=True
)


# Step 4: Three search queries

queries = [
    "I was sick before the deadline. Can I submit my assignment later?",
    "I cannot log into my student account. How can I get access again?",
    "My printer is producing blank pages. What should I check?"
]


# Step 5: Display search results

def show_results(query, scores, method):

    print("\nSearch Method:", method)
    print("Query:", query)

    # Sort scores from highest to lowest
    ranked = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )

    # Display top 3 results
    for i in ranked[:3]:

        print("\nChunk ID:", records[i]["id"])
        print("Source:", records[i]["source"])
        print("Score:", round(float(scores[i]), 4))

        print("Original Evidence:")
        print(records[i]["text"])


# Step 6: Compare TF-IDF and Embeddings

for query in queries:

    print("\n")
    print("NEW SEARCH")

    # TF-IDF search
    query_data = vectorizer.transform([query])

    tfidf_scores = cosine_similarity(
        query_data,
        tfidf_data
    )[0]

    show_results(query, tfidf_scores, "TF-IDF")

    # Embedding search
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    embedding_scores = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    show_results(query, embedding_scores, "Embeddings")


print("\nAll searches completed successfully!")

Total chunks: 21


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]



NEW SEARCH

Search Method: TF-IDF
Query: I was sick before the deadline. Can I submit my assignment later?

Chunk ID: handbook:3
Source: handbook.txt
Score: 0.327
Original Evidence:
If the portal is unavailable, students should take a screenshot of the error and email the course coordinator with the assignment attached before the deadline. The coordinator can confirm whether the submission time will be adjusted. 

Chunk ID: handbook:1
Source: handbook.txt
Score: 0.2687
Original Evidence:
Late submissions are normally penalized, but students who become ill before a deadline may request an extension if they contact the coordinator before the deadline and provide medical evidence. 

Chunk ID: handbook:0
Source: handbook.txt
Score: 0.2183
Original Evidence:
University assignments should normally be submitted through the learning portal before the deadline shown on the course page. 

Search Method: Embeddings
Query: I was sick before the deadline. Can I submit my assignment later?

Chunk 